## 第26章　线性回归 —— 全流程闭环

> 本章目标：用两种方法实现线性回归——闭式解（第 7 章逆矩阵）和梯度下降（第 12 章 SGD）——完整对比它们的耗时、精度和适用场景。这是全书第一个"从数据生成到模型拟合"的全流程闭环，也是理解"训练"概念的最小完整示例。
> 前置知识：第 7 章（逆矩阵/正规方程）、第 12 章（梯度下降）、第 17 章（期望与方差）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")



### 26.1　生成合成数据 —— 从真实参数到带噪声的观测

真实世界的数据来自某个未知的"生成过程"。对线性回归来说，这个过程极其简单：`y = w·x + b + noise`。我们从已知的真实参数出发（w=3, b=−2），加入高斯噪声模拟测量误差，生成 100 个观测点。

💻 **代码　生成 y = 3x − 2 + 噪声的合成数据**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 100
true_w, true_b = 3.0, -2.0
X_raw = np.random.uniform(-3, 3, size=n)
y = true_w * X_raw + true_b + np.random.randn(n) * 1.5

plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.5, s=15, label='Observed data')
x_line = np.linspace(-3, 3, 100)
plt.plot(x_line, true_w * x_line + true_b, 'g--', lw=2, label=f'True: y={true_w}x{true_b:+.0f}')
plt.xlabel('x'); plt.ylabel('y'); plt.title('Synthetic Linear Regression Data')
plt.legend(); plt.grid(alpha=0.3); plt.show()




---

### 26.2　闭式解 —— 一行 NumPy 秒算最优参数

第 7 章推导的正规方程在这里直接落地：`w = (XᵀX)⁻¹ Xᵀ y`。构建设计矩阵 X（加一列全 1 代表截距），一行 NumPy 解出最优权重——不需要学习率、不需要迭代、不需要担心收敛。

📐 **定义　正规方程（Normal Equation）**：w = (XᵀX)⁻¹ Xᵀ y。复杂度 O(d³)，d 是特征数。小数据（d<10⁴）最优，大数据不可用（求逆太慢）。

💻 **代码　闭式解：一行 NumPy**




In [ ]:
import numpy as np

np.random.seed(42)
n = 100
X_raw = np.random.uniform(-3, 3, size=n)
y = 3.0 * X_raw - 2.0 + np.random.randn(n) * 1.5

# 构建设计矩阵：[x, 1]
X = np.column_stack([X_raw, np.ones(n)])
# 闭式解：一行！
w_closed = np.linalg.inv(X.T @ X) @ X.T @ y

print(f"真实参数: w=3.0, b=-2.0")
print(f"闭式解:   w={w_closed[0]:.3f}, b={w_closed[1]:.3f}")




> **关键洞察**：`(XᵀX)⁻¹ Xᵀ y`——四个矩阵操作，不需要 for 循环。这就是闭式解的威力：精确、无需调参、O(d³) 时间内达到数学最优。

---

### 26.3　梯度下降解 —— 迭代逼近最优

同样的数据，换一种思路：从随机初始化的参数出发，每一步沿负梯度方向走一小步。核心更新：`w -= lr * (2/n) * Xᵀ @ (X @ w − y)`。

📐 **梯度下降 vs 闭式解**：闭式解精度高但复杂度 O(d³)，梯度下降可处理百万特征但需要调学习率 + 迭代次数。

💻 **代码　梯度下降：500 步逼近最优**




In [ ]:
import numpy as np

np.random.seed(42)
n = 100
X_raw = np.random.uniform(-3, 3, size=n)
X = np.column_stack([X_raw, np.ones(n)])
y = 3.0 * X_raw - 2.0 + np.random.randn(n) * 1.5

# 梯度下降
w_gd = np.array([0.0, 0.0])
lr = 0.01
for epoch in range(500):
    grad = (2/n) * X.T @ (X @ w_gd - y)
    w_gd -= lr * grad

print(f"真实参数: w=3.0, b=-2.0")
print(f"梯度下降: w={w_gd[0]:.3f}, b={w_gd[1]:.3f}")




---

### 26.4　对比两种方法 —— 耗时、精度与适用场景

💻 **代码　闭式解 vs 梯度下降：同数据，两条路**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)
n = 100
X_raw = np.random.uniform(-3, 3, size=n)
y = 3.0 * X_raw - 2.0 + np.random.randn(n) * 1.5
X = np.column_stack([X_raw, np.ones(n)])

# 闭式解
t0 = time.perf_counter()
w_closed = np.linalg.inv(X.T @ X) @ X.T @ y
t1 = time.perf_counter()

# 梯度下降
w_gd = np.array([0.0, 0.0])
t2 = time.perf_counter()
for _ in range(500):
    w_gd -= 0.01 * (2/n) * X.T @ (X @ w_gd - y)
t3 = time.perf_counter()

print(f"闭式解:   w={w_closed[0]:.3f}, b={w_closed[1]:.3f} ({t1-t0:.6f}s)")
print(f"梯度下降: w={w_gd[0]:.3f}, b={w_gd[1]:.3f} ({t3-t2:.4f}s)")
print(f"两者一致: {np.allclose(w_closed, w_gd, atol=0.05)}")

# 可视化
x_line = np.linspace(-3, 3, 100)
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.5, s=15, label='Data')
plt.plot(x_line, w_closed[0]*x_line + w_closed[1], 'r-', lw=2, label='Closed-form')
plt.plot(x_line, w_gd[0]*x_line + w_gd[1], 'b--', lw=2, label='Gradient Descent')
plt.xlabel('x'); plt.ylabel('y'); plt.title('Linear Regression: Closed-form vs GD')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print("\n闭式解: 小数据(d<10000)最优——精确、无需调参")
print("梯度下降: 大数据/深度学习标配——O(nd)可扩展")




> **关键洞察**：两种方法给出几乎相同的拟合线。但闭式解的 O(d³) 复杂度意味着特征数上万时求逆已不可行——这正是梯度下降的用武之地。在深度学习中你永远见不到闭式解——因为对亿×亿矩阵求逆在物理上不可能。

🔗 **AI 连接**：第 28 章的两层网络本质上就是两次线性变换中间夹一个 ReLU——梯度下降是唯一可行的训练方式。第 24 章的 Adam 等优化器就是在梯度下降基础上的工程升级。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）闭式解 vs 梯度下降各适用于什么场景？复杂度分别是多少？
2. （代码）生成 y=2x+5+noise 的 200 个点，用闭式解和梯度下降分别拟合，打印参数并画图对比。
3. （代码）保持真实参数不变，逐渐增大样本量 n=[10, 100, 1000]，对比闭式解的参数估计精度（与真实值的差距）。观察大数定律在参数估计中的体现。
---
> 预览：下一章——**逻辑回归**，概率视角的分类器。
